# Model comparison

Trains one YOLO instance segmentation model on the grape dataset, as in the
first two experimental stages: YOLOv11 and YOLOv26, with and without offline
and online data augmentation, in different model sizes. Run one configuration
per session.

## Setup

Select a GPU runtime (**Runtime > Change runtime type**). Outputs are written
to Google Drive, so that a run interrupted by the end of the session can be
resumed. The Roboflow API key is read from the Colab secret
`ROBOFLOW_API_KEY` (key icon in the left panel).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/hugocesarleal/grape-segmentation-training /content/grape-segmentation-training
%cd /content/grape-segmentation-training
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
OUTPUT = "/content/drive/MyDrive/grape-segmentation"
os.makedirs(OUTPUT, exist_ok=True)
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

## Dataset

Version 14 is the export without offline augmentation. The step also converts
the instances exported as boxes into polygons.

In [ ]:
VERSION = 14  # @param {type:"integer"}
!python -m grapeseg.data --version $VERSION --out /content/dataset

## Training

Use the dataset version with offline augmentation together with
`OFFLINE = True`. The batch size is reduced by Ultralytics if the GPU memory is
insufficient.

In [ ]:
MODEL = "yolo26l-seg.pt"  # @param ["yolo11s-seg.pt", "yolo26s-seg.pt", "yolo26m-seg.pt", "yolo26l-seg.pt", "yolo26x-seg.pt"]
OFFLINE = False  # @param {type:"boolean"}
ONLINE = True  # @param {type:"boolean"}
BATCH = 8  # @param {type:"integer"}
NAME = "yolo26l-online"  # @param {type:"string"}

flags = ("--offline " if OFFLINE else "") + ("" if ONLINE else "--no-online")
!python train.py --data /content/dataset/data.yaml --model $MODEL --batch $BATCH --name $NAME --project $OUTPUT $flags

## Resume an interrupted run

Run the setup and dataset cells again, then this cell.

In [ ]:
!python train.py --resume $OUTPUT/$NAME/weights/last.pt